In [ ]:
# Install the required libraries.

%pip install configobj
%pip install simpleitk
%pip install numpy
%pip install --upgrade scikit-learn
%pip install matplotlib
%pip install scipy
%pip install nibabel
%pip install scikit-learn
%pip install tensorflow

In [ ]:
import SimpleITK as sitk
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from scipy.linalg import orthogonal_procrustes
from scipy.spatial.transform import Rotation
from scipy.interpolate import griddata
import csv

segmentation_paths = {
    "core": {
        "label": "3-month",
        "path": "./3_month_patient1/3Month25Labels_Stealth.nii"
    },
    "comparisons": [
        {
            "label": "30-days",
            "path": "./30_Days_patient1/30days_patient1_SAG_ST_25implants.nii"
        }
    ]
}

def load_data(segmentation_path):
    print(f"Loading NIfTI file from {segmentation_path}...")
    image = sitk.ReadImage(segmentation_path)
    data = sitk.GetArrayFromImage(image)
    spacing = image.GetSpacing()
    print("NIfTI file loaded successfully.")
    print(f"Spacing (x, y, z): {spacing}") 
    return data, spacing

In [ ]:
data, spacing = load_data("./data/3_month_patient1/3Month25Labels_Stealth.nii")

In [ ]:
data, spacing = load_data("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load the NIfTI file
image = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")  # update with your path
data = sitk.GetArrayFromImage(image)
spacing = image.GetSpacing()

# Define control labels
control_labels = [2, 3, 4, 5, 7]
spacing = spacing[::-1]  # flip to (z, y, x)

# Gather physical coordinates
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for label in control_labels:
    coords = np.argwhere(data == label)
    if coords.size > 0:
        coords_mm = coords * spacing
        ax.scatter(coords_mm[:, 2], coords_mm[:, 1], coords_mm[:, 0], label=f'Label {label}', alpha=0.6)

ax.set_title("Control Label Locations (in mm)")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_zlabel("Z (mm)")
ax.legend()
plt.show()


In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load your NIfTI file
image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")  # update with your path
data1 = sitk.GetArrayFromImage(image1)
spacing1 = image1.GetSpacing()[::-1]  # Flip to (z, y, x) to match np.argwhere

#another set
image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")
data2 = sitk.GetArrayFromImage(image2)
spacing2 = image2.GetSpacing()[::-1]

# Choose labels of interest (example: 11 to 25)
labels_of_interest = list(range(1, 26))

# Plot setup
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

for label in labels_of_interest:
    coords = np.argwhere(data1 == label)
    if coords.shape[0] == 0:
        print(f"Label {label}: no voxels found.")
        continue
    physical_coords = coords * spacing1
    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],
               label=f'Label {label}', s=1, alpha=0.5)

ax.set_title("3D Plot of Anatomical Labels (Physical Space in mm)")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_zlabel("Z (mm)")
ax.legend(markerscale=4, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def reorient_points(points, axis_order=[0, 1, 2], flip_axes=[1, 1, 1]):
    """
    Reorder and optionally flip axes of a 3D point cloud.

    Args:
        points: NumPy array of shape (N, 3)
        axis_order: List of length 3 indicating axis permutation. 
                    E.g., [2, 1, 0] swaps from (z, y, x) to (x, y, z)
        flip_axes: List of 1 or -1 to flip each axis. E.g., [1, -1, 1] flips y-axis.

    Returns:
        Transformed points (N, 3)
    """
    reordered = points[:, axis_order]
    flipped = reordered * flip_axes
    return flipped

def plot_points(points, title="3D Point Cloud", label=None):
    """
    Simple 3D scatter plot for visualizing coordinates.

    Args:
        points: NumPy array of shape (N, 3)
        title: Plot title
        label: Optional point label
    """
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], alpha=0.6, s=1, label=label)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title)
    if label:
        ax.legend()
    plt.show()



In [ ]:
# Choose labels of interest (example: 11 to 25)
labels_of_interest = list(range(1, 26))

# Plot setup
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

for label in labels_of_interest:
    coords = np.argwhere(data2 == label)
    aligned2 = reorient_points(coords, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])
    if aligned2.shape[0] == 0:
        print(f"Label {label}: no voxels found.")
        continue
    physical_coords = aligned2 * spacing1
    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],
               label=f'Label {label}', s=1, alpha=0.5)

ax.set_title("3D Plot of Anatomical Labels (Physical Space in mm)")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_zlabel("Z (mm)")
ax.legend(markerscale=4, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
coords = np.argwhere(data1 == 1)  # Get label 1 voxels


# Convert to real-world mm
coords_mm = coords * np.array(spacing1[::-1])


aligned1 = reorient_points(coords_mm, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])
plot_points(aligned1, title="Label 1 (Reoriented)", label="Label 1")


In [ ]:
import numpy as np

coords_label1 = np.argwhere(data1 == 1)

coords_label1_mm = coords_label1 * np.array(spacing1[::-1])

print(coords_label1_mm)

In [ ]:
coords = np.argwhere(data2 == 1)  # Get label 1 voxels


# Convert to real-world mm
coords_mm = coords * np.array(spacing2[::-1])

# Try reorienting: from (z, y, x) → (x, y, z)
aligned2 = reorient_points(coords_mm, axis_order=[1, 2, 0], flip_axes=[1, 1, 1])
plot_points(aligned2, title="Label 11 (Reoriented)", label="Label 11")

In [ ]:
import numpy as np

coords_label2 = np.argwhere(data2 == 1)

coords_label2_mm = coords_label2 * np.array(spacing2[::-1])

print(coords_label2_mm)

In [ ]:
import numpy as np

direction = image1.GetDirection()

direction_matrix = np.array(direction).reshape(3, 3)
print("3x3 Direction matrix:\n", direction_matrix)

In [ ]:
import numpy as np

def get_label_centroid(data, label, spacing):
    coords = np.argwhere(data == label)
    if coords.size == 0:
        return None
    coords_mm = coords * np.array(spacing[::-1])
    return np.mean(coords_mm, axis=0)

def compare_centroids(data1, data2, spacing1, spacing2, labels):
    print("Comparing centroids (in mm):")
    for label in labels:
        c1 = get_label_centroid(data1, label, spacing1)
        c2 = get_label_centroid(data2, label, spacing2)
        if c1 is None or c2 is None:
            print(f"Label {label}: not found in one of the datasets.")
            continue
        diff = np.linalg.norm(c1 - c2)
        print(f"Label {label}: distance = {diff:.2f} mm | Δx = {c1[0]-c2[0]:.2f}, Δy = {c1[1]-c2[1]:.2f}, Δz = {c1[2]-c2[2]:.2f}")


In [ ]:
def dice_score(mask1, mask2):
    intersection = np.sum((mask1 > 0) & (mask2 > 0))
    size1 = np.sum(mask1 > 0)
    size2 = np.sum(mask2 > 0)
    if size1 + size2 == 0:
        return 1.0
    return 2.0 * intersection / (size1 + size2)

def compare_dice(data1, data2, labels):
    print("\nDice similarity scores:")
    for label in labels:
        mask1 = (data1 == label).astype(np.uint8)
        mask2 = (data2 == label).astype(np.uint8)
        score = dice_score(mask1, mask2)
        print(f"Label {label}: Dice = {score:.3f}")


In [ ]:
# Show both labels in 3D
coords1 = np.argwhere(data1 == 11) * spacing1[::-1]
coords2 = np.argwhere(data2 == 11) * spacing2[::-1]

plot_points(aligned1, title="Label 11 - Scan A", label="Scan A")
plot_points(aligned2, title="Label 11 - Scan B", label="Scan B")


In [ ]:
compare_centroids(data2, data1, spacing2, spacing1, labels_of_interest)


In [ ]:
# Load your NIfTI file
image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")  # update with your path
data1 = sitk.GetArrayFromImage(image1)
spacing1 = image1.GetSpacing()[::-1]  # Flip to (z, y, x) to match np.argwhere

#another set
image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")
data2 = sitk.GetArrayFromImage(image2)
spacing2 = image2.GetSpacing()[::-1]

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
 
# Load your NIfTI files
image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")
data1 = sitk.GetArrayFromImage(image1)
spacing1 = image1.GetSpacing()[::-1]  # Flip to (z, y, x) for np.argwhere
 
image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")
data2 = sitk.GetArrayFromImage(image2)
spacing2 = image2.GetSpacing()[::-1]
 
# Choose labels of interest
labels_of_interest = list(range(1, 26))
 
# Plot setup
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
 
for label in labels_of_interest:
    coords = np.argwhere(data1 == label)
    if coords.shape[0] == 0:
        print(f"Label {label}: no voxels found.")
        continue
    physical_coords = coords * spacing1
    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],
               label=f'Label {label}', s=1, alpha=0.5)
 
ax.set_title("3D Plot of Anatomical Labels (Physical Space in mm)")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_zlabel("Z (mm)")
ax.legend(markerscale=4, bbox_to_anchor=(1.05, 1), loc='upper left')
 
plt.tight_layout()
plt.show()
 
# ✅ Save view/camera settings for reuse in Cell 10
elev = ax.elev
azim = ax.azim
xlim = ax.get_xlim()
ylim = ax.get_ylim()
zlim = ax.get_zlim()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
 
def reorient_points(points, axis_order=[0, 1, 2], flip_axes=[1, 1, 1]):
    """
    Reorder and optionally flip axes of a 3D point cloud.
 
    Args:
        points: NumPy array of shape (N, 3)
        axis_order: List of length 3 indicating axis permutation. 
                    E.g., [2, 1, 0] swaps from (z, y, x) to (x, y, z)
        flip_axes: List of 1 or -1 to flip each axis. E.g., [1, -1, 1] flips y-axis.
 
    Returns:
        Transformed points (N, 3)
    """
    reordered = points[:, axis_order]
    flipped = reordered * flip_axes
    return flipped
 
def plot_points(points, title="3D Point Cloud", label=None,
                elev=None, azim=None, xlim=None, ylim=None, zlim=None):
    """
    Simple 3D scatter plot for visualizing coordinates, with optional view alignment.
 
    Args:
        points: NumPy array of shape (N, 3)
        title: Plot title
        label: Optional point label
        elev, azim: Optional camera angles to match another plot
        xlim, ylim, zlim: Optional axis limits to match another plot
    """
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], alpha=0.6, s=1, label=label)
 
    # Apply axis labels and title
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title)
 
    if label:
        ax.legend()
 
    # ✅ Optional: Apply same view if given
    if elev is not None and azim is not None:
        ax.view_init(elev=elev, azim=azim)
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)
    if zlim: ax.set_zlim(zlim)
 
    plt.show()

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def reorient_points(points, axis_order=[2, 1, 0], flip_axes=[0, 0, 0]):
    """
    Reorient voxel points using axis permutation and flips.
    """
    # Permute
    reoriented = points[:, axis_order]
    # Flip axes
    for i, flip in enumerate(flip_axes):
        if flip:
            reoriented[:, i] = -reoriented[:, i]
    return reoriented

# --- Load NIfTI images ---
image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")
data1 = sitk.GetArrayFromImage(image1)
spacing1 = image1.GetSpacing()[::-1]  # (z, y, x)

image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")
data2 = sitk.GetArrayFromImage(image2)
spacing2 = image2.GetSpacing()[::-1]

# --- Choose label range ---
labels_of_interest = list(range(1, 26))

# --- Plot setup ---
fig = plt.figure(figsize=(14, 12))
ax = fig.add_subplot(111, projection='3d')

# --- Plot data1 in blue ---
for label in labels_of_interest:
    coords = np.argwhere(data1 == label)
    if coords.size == 0:
        continue
    physical_coords = coords * spacing1
    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],
               label=f'data1 - Label {label}', s=1, alpha=0.4, c='blue')

# --- Plot data2 in red ---
for label in labels_of_interest:
    coords = np.argwhere(data2 == label)
    if coords.size == 0:
        continue
    aligned_coords = reorient_points(coords, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])  # Adjust as needed
    physical_coords = aligned_coords * spacing1  # Using spacing1 to align scales
    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],
               label=f'data2 - Label {label}', s=1, alpha=0.4, c='red')

# --- Final plot labels ---
ax.set_title("3D Comparison of Anatomical Labels (data1 in blue, data2 in red)")
ax.set_xlabel("X (mm)")
ax.set_ylabel("Y (mm)")
ax.set_zlabel("Z (mm)")
ax.legend(markerscale=3, loc='upper left', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D

import numpy as np
 
# Labels to visualize

labels_of_interest = list(range(1, 26))  # Update if needed
 
# Create side-by-side subplots

fig = plt.figure(figsize=(20, 8))
 
# 🔵 LEFT: 3-Month Scan

ax1 = fig.add_subplot(121, projection='3d')

for label in labels_of_interest:

    coords = np.argwhere(data1 == label)

    if coords.shape[0] == 0:

        continue

    physical_coords = coords * spacing1

    ax1.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],

                s=1, alpha=0.5, c='blue', label='3-Month' if label == 1 else None)
 
ax1.set_title("3-Month Implant Labels")

ax1.set_xlabel("X (mm)")

ax1.set_ylabel("Y (mm)")

ax1.set_zlabel("Z (mm)")

ax1.legend(markerscale=5)
 
# 🔴 RIGHT: 30-Day Scan (Reoriented)

ax2 = fig.add_subplot(122, projection='3d')

for label in labels_of_interest:

    coords = np.argwhere(data2 == label)

    if coords.shape[0] == 0:

        continue

    aligned_coords = reorient_points(coords, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])

    physical_coords = aligned_coords * spacing2

    ax2.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],

                s=1, alpha=0.5, c='red', label='30-Day' if label == 1 else None)
 
ax2.set_title("30-Day Implant Labels (Reoriented)")

ax2.set_xlabel("X (mm)")

ax2.set_ylabel("Y (mm)")

ax2.set_zlabel("Z (mm)")

ax2.legend(markerscale=5)
 
# ✅ Optional: Match camera views if you like (copy from one to the other)

ax2.view_init(elev=ax1.elev, azim=ax1.azim)
 
plt.tight_layout()

plt.show()

 

In [ ]:
label_centroids_3m = {}

label_centroids_30d = {}

diff_vectors = []
 
labels_of_interest = list(range(1, 26))
 
# Step 1: Get centroids for each label

for label in labels_of_interest:

    coords_3m = np.argwhere(data1 == label)

    coords_30d = np.argwhere(data2 == label)
 
    if coords_3m.shape[0] == 0 or coords_30d.shape[0] == 0:

        print(f"Label {label} missing in one of the datasets.")

        continue
 
    centroid_3m = np.mean(coords_3m * spacing1, axis=0)
 
    aligned_30d = reorient_points(coords_30d, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])

    centroid_30d = np.mean(aligned_30d * spacing2, axis=0)
 
    label_centroids_3m[label] = centroid_3m

    label_centroids_30d[label] = centroid_30d
 
    diff = centroid_3m - centroid_30d

    diff_vectors.append(diff)
 
# Step 2: Compute average translation vector

translation_vector = np.mean(diff_vectors, axis=0)

print("Computed translation vector:", translation_vector)
 
# Step 3: Overlay plot with translation applied

fig = plt.figure(figsize=(12, 10))

ax = fig.add_subplot(111, projection='3d')
 
# 🔵 Plot 3-month

for label in labels_of_interest:

    coords = np.argwhere(data1 == label)

    if coords.shape[0] == 0:

        continue

    physical_coords = coords * spacing1

    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],

               s=1, alpha=0.5, c='blue', label='3-Month' if label == 1 else None)
 
# 🔴 Plot aligned 30-day (after translation)

for label in labels_of_interest:

    coords = np.argwhere(data2 == label)

    if coords.shape[0] == 0:

        continue

    aligned = reorient_points(coords, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])

    physical_coords = (aligned * spacing2) + translation_vector  # ✅ Apply translation

    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],

               s=1, alpha=0.5, c='red', label='30-Day (Aligned)' if label == 1 else None)
 
# Final setup

ax.set_title("Overlay After Centroid-Based Alignment")

ax.set_xlabel("X (mm)")

ax.set_ylabel("Y (mm)")

ax.set_zlabel("Z (mm)")

ax.legend(markerscale=5, bbox_to_anchor=(1.05, 1), loc='upper left')
 
plt.tight_layout()

plt.show()

 

In [ ]:
import matplotlib.pyplot as plt

from mpl_toolkits.mplot3d import Axes3D

import numpy as np
 
label_centroids_3m = {}

label_centroids_30d = {}

diff_vectors = []
 
labels_of_interest = list(range(1, 26))

shift_distances = {}
 
# Step 1: Compute centroids and diff vectors

for label in labels_of_interest:

    coords_3m = np.argwhere(data1 == label)

    coords_30d = np.argwhere(data2 == label)
 
    if coords_3m.shape[0] == 0 or coords_30d.shape[0] == 0:

        print(f"Label {label}: missing in one of the datasets.")

        continue
 
    centroid_3m = np.mean(coords_3m * spacing1, axis=0)
 
    aligned_30d = reorient_points(coords_30d, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])

    centroid_30d = np.mean(aligned_30d * spacing2, axis=0)
 
    label_centroids_3m[label] = centroid_3m

    label_centroids_30d[label] = centroid_30d
 
    diff = centroid_3m - centroid_30d

    diff_vectors.append(diff)
 
    # Calculate Euclidean distance per label

    distance = np.linalg.norm(diff)

    shift_distances[label] = distance

    print(f"Label {label} shift distance: {distance:.2f} mm")
 
# Step 2: Compute mean translation vector

translation_vector = np.mean(diff_vectors, axis=0)

print("\n▶️ Mean translation vector applied to 30-day data:", translation_vector)
 
# Step 3: Plot everything

fig = plt.figure(figsize=(12, 10))

ax = fig.add_subplot(111, projection='3d')
 
# 🔵 Plot 3-month data

for label in labels_of_interest:

    coords = np.argwhere(data1 == label)

    if coords.shape[0] == 0:

        continue

    physical_coords = coords * spacing1

    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],

               s=1, alpha=0.5, c='blue', label='3-Month' if label == 1 else None)
 
# 🔴 Plot aligned 30-day data (after applying translation)

for label in labels_of_interest:

    coords = np.argwhere(data2 == label)

    if coords.shape[0] == 0:

        continue

    aligned = reorient_points(coords, axis_order=[2, 1, 0], flip_axes=[1, 1, 1])

    physical_coords = (aligned * spacing2) + translation_vector

    ax.scatter(physical_coords[:, 2], physical_coords[:, 1], physical_coords[:, 0],

               s=1, alpha=0.5, c='red', label='30-Day (Aligned)' if label == 1 else None)
 
# ✏️ Draw arrows from 30-day → 3-month centroids

for label in label_centroids_3m:

    start = label_centroids_30d[label] + translation_vector

    end = label_centroids_3m[label]

    arrow = end - start

    ax.quiver(start[2], start[1], start[0],

              arrow[2], arrow[1], arrow[0],

              color='green', arrow_length_ratio=0.1, linewidth=1)
 
# 🧭 Axis Labels

ax.set_title("Centroid-Based Alignment with Arrows and Shift Distances")

ax.set_xlabel("X (mm)")

ax.set_ylabel("Y (mm)")

ax.set_zlabel("Z (mm)")

ax.legend(markerscale=5, bbox_to_anchor=(1.05, 1), loc='upper left')
 
plt.tight_layout()

plt.show()

 

In [ ]:
import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt

# Load label images using actual file paths

image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")
image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")

# Compute centroids (physical coordinates) for labels 1..25 in each image
stats1 = sitk.LabelShapeStatisticsImageFilter()
stats2 = sitk.LabelShapeStatisticsImageFilter()
stats1.Execute(image1)
stats2.Execute(image2)
labels = range(1, 26)
centroids1 = [stats1.GetCentroid(l) for l in labels if stats1.HasLabel(l)]
centroids2 = [stats2.GetCentroid(l) for l in labels if stats2.HasLabel(l)]
centroids1 = np.array(centroids1)
centroids2 = np.array(centroids2)

# Compute global means of centroids in each image
mean1 = centroids1.mean(axis=0)
mean2 = centroids2.mean(axis=0)

# Rigid transform (Kabsch algorithm) aligning image2->image1 centroids
P = centroids1 - mean1
Q = centroids2 - mean2
H = Q.T.dot(P)
U, S, Vt = np.linalg.svd(H)
R = Vt.T.dot(U.T)
if np.linalg.det(R) < 0:
    Vt[-1, :] *= -1
    R = Vt.T.dot(U.T)
t = mean1 - R.dot(mean2)

print("Estimated rotation matrix R:")
print(R)
print("Estimated translation vector t:")
print(t)

# Apply transform to image2 centroids
centroids2_transformed = (R.dot(centroids2.T)).T + t

# 2D Visualization in Y-Z plane
# Create subplots for XY, YZ, and XZ views
fig, axs = plt.subplots(1, 3, figsize=(18, 6))

views = {
    "XY View": (0, 1),
    "YZ View": (1, 2),
    "XZ View": (0, 2)
}

colors = ['red', 'blue']
labels_text = ['3 Month', '30 Day']

for ax, (title, (i, j)) in zip(axs, views.items()):
    for idx, label in enumerate(labels):
        if idx >= len(centroids1) or idx >= len(centroids2_transformed):
            continue
        x1, y1 = centroids1[idx, i], centroids1[idx, j]
        x2, y2 = centroids2_transformed[idx, i], centroids2_transformed[idx, j]

        # Scatter points
        ax.scatter(x1, y1, color=colors[0], s=40)
        ax.scatter(x2, y2, color=colors[1], s=40)

        # Annotate points with label number
        ax.text(x1 , y1 , str(label), color=colors[0], fontsize=8)
        ax.text(x2 , y2 , str(label), color=colors[1], fontsize=8)

    ax.set_xlabel(["X", "Y", "Z"][i])
    ax.set_ylabel(["X", "Y", "Z"][j])
    ax.set_title(title)


# Avoid duplicate labels in legend
handles, labels = axs[0].get_legend_handles_labels()
unique = dict(zip(labels, handles))
axs[0].legend(unique.values(), unique.keys(), loc="upper right")

plt.tight_layout()
plt.show()



In [ ]:
def get_label_stats(image, label_id):
    array = sitk.GetArrayFromImage(image)
    indices = np.argwhere(array == label_id)  # (z, y, x) format
    if indices.size == 0:
        return None

    # Convert to physical coordinates
    image_spacing = image.GetSpacing()
    image_origin = image.GetOrigin()
    image_direction = np.array(image.GetDirection()).reshape(3, 3)

    # Convert voxel indices (z, y, x) to physical (x, y, z)
    coords = []
    for z, y, x in indices:
        physical = np.array([x, y, z]) * image_spacing[::-1]  # flip spacing to (x,y,z)
        physical = image_direction @ physical + image_origin
        coords.append(physical)

    coords = np.array(coords)
    bbox_min = coords.min(axis=0)
    bbox_max = coords.max(axis=0)
    variance = np.var(coords, axis=0)
    return {
        'variance': variance,
        'bbox_min': bbox_min,
        'bbox_max': bbox_max,
        'coords': coords
    }


In [ ]:

image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")
image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")

for label_id in range(1, 26):
    stat1 = get_label_stats(image1, label_id)
    stat2 = get_label_stats(image2, label_id)
    if stat1 is None or stat2 is None:
        continue

    print(f"\nLabel {label_id}")
    print(f"3 Month Bounding Box: min={stat1['bbox_min']}, max={stat1['bbox_max']}")
    print(f"30 Day Bounding Box: min={stat2['bbox_min']}, max={stat2['bbox_max']}")
    print(f"3 Month Variance: {stat1['variance']}")
    print(f"30 Day Variance: {stat2['variance']}")


In [ ]:
def voxel_coords(image, label_id):
    array = sitk.GetArrayFromImage(image)
    indices = np.argwhere(array == label_id)  # (z, y, x)
    if indices.size == 0:
        return None

    spacing = np.array(image.GetSpacing())  # in (x, y, z)
    origin = np.array(image.GetOrigin())
    direction = np.array(image.GetDirection()).reshape(3, 3)

    # Convert voxel indices (z, y, x) to (x, y, z) then to physical
    coords = []
    for z, y, x in indices:
        voxel = np.array([x, y, z])
        physical = direction @ (voxel * spacing) + origin
        coords.append(physical)
    return np.array(coords)



In [ ]:
def apply_rigid_transform(coords, R, t):
    return (R @ coords.T).T + t

def bounding_box(coords):
    return coords.min(axis=0), coords.max(axis=0)

def count_inside(coords_to_check, bbox_min, bbox_max):
    inside = np.all((coords_to_check >= bbox_min) & (coords_to_check <= bbox_max), axis=1)
    return np.sum(inside)


In [ ]:
#image1 = sitk.ReadImage('./data/1_month_patient2/patient2_1month_CTLOCALIZATION.nii')
#image2 = sitk.ReadImage('./data/2_week_patient2/patient2_2week_Sagital.nii')

#image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth_controlled.nii")
#image2 = sitk.ReadImage("./data/30_days_patient1/30DAY_IMG00001_CTSTealthBone_controlled.nii")

image1 = sitk.ReadImage("./data/3_month_patient1/3Month25Labels_Stealth.nii")
image2 = sitk.ReadImage("./data/30_Days_patient1/ser00006/30days_patient1_SAG ST_25implants.nii")

#image1 = sitk.ReadImage("./data/3_month_patient1/Patient-1_3-Month-Follow-up_1(SAG MPR BONE).nii")
#image2 = sitk.ReadImage("./data/30_days_patient1/Patient_1_30 Day_SER00006_IMG00001.nii")

#image1 = sitk.ReadImage('./data/1_month_patient2/Patient-2_1-Month_CORONAL-MPR-BONE.nii')
#image2 = sitk.ReadImage('./data/2_week_patient2/Patient-2_2-Weeks_COR-BONE.nii')

for label in range(1, 26):
    coords1 = voxel_coords(image1, label)  
    coords2 = voxel_coords(image2, label) 

    if coords1 is None or coords2 is None:
        print(f"Label {label}: Skipped (not found in both images)")
        continue

    # Step 1: Transform 30-day coordinates to align with 3-month space
    coords2_transformed = apply_rigid_transform(coords2, R, t)

    # Step 2: Bounding box from 3-month label
    bbox_min, bbox_max = bounding_box(coords1)

    # Step 3: Count how many transformed voxels are inside 3-month bbox
    count = count_inside(coords2_transformed, bbox_min, bbox_max)
    percent = 100 * count / len(coords2)

    print(f"Label {label}: {count} voxel(s) ({percent:.2f}%) from 30-day fall inside 3-month bounding box **after transform**")


In [ ]:
from scipy.spatial.distance import cdist

def compute_distances(coords1, coords2_transformed):
    # Centroid distance
    centroid1 = coords1.mean(axis=0)
    centroid2 = coords2_transformed.mean(axis=0)
    centroid_distance = np.linalg.norm(centroid1 - centroid2)

    # Minimum voxel-to-voxel distance
    distances = cdist(coords1, coords2_transformed)
    min_distance = np.min(distances)

    return centroid_distance, min_distance


In [ ]:
for label in range(1, 26):
    coords1 = voxel_coords(image1, label)  
    coords2 = voxel_coords(image2, label) 

    if coords1 is None or coords2 is None:
        print(f"Label {label}: Skipped (not found in both images)")
        continue

    # Apply transformation
    coords2_transformed = apply_rigid_transform(coords2, R, t)

    # Bounding box check
    bbox_min, bbox_max = bounding_box(coords1)
    count = count_inside(coords2_transformed, bbox_min, bbox_max)
    percent = 100 * count / len(coords2)

    # Distance calculations
    centroid_dist, min_voxel_dist = compute_distances(coords1, coords2_transformed)

    print(f"Label {label}:")
    print(f" - {count} voxel(s) ({percent:.2f}%) inside bounding box after transform")
    print(f" - Centroid Distance: {centroid_dist:.2f} mm")
    print(f" - Minimum Voxel Distance: {min_voxel_dist:.2f} mm")


In [ ]:
pip install torch

In [ ]:
import nibabel as nib
import numpy as np
import torch
from torch import nn

# Load a NIfTI image
nii = nib.load("./data/3_month_patient1/3Month25Labels_Stealth.nii")
img = nii.get_fdata()
img = (img - img.min()) / (img.max() - img.min())

# Flatten image and create coordinate grid
D, H, W = img.shape
coords = np.stack(np.meshgrid(np.linspace(0, 1, D),
                              np.linspace(0, 1, H),
                              np.linspace(0, 1, W), indexing='ij'), -1).reshape(-1, 3)
intensities = img.flatten()

# Define implicit neural representation (MLP)
class INR(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.net(x)

# Train using prior and fine-tune on sparse data


In [ ]:
import nibabel as nib
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import random

class NeRPDataset(Dataset):
    def __init__(self, coords, intensities):
        self.coords = torch.tensor(coords, dtype=torch.float32)
        self.intensities = torch.tensor(intensities, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        return self.coords[idx], self.intensities[idx]

class NeRPMLP(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)

# Load and preprocess a NIfTI image
img = nib.load("./data/3_month_patient1/3Month25Labels_Stealth.nii").get_fdata()
img = (img - np.min(img)) / (np.max(img) - np.min(img))

# Downsample and generate coordinate-intensity pairs
img = img[::2, ::2, ::2]
D, H, W = img.shape
coords = np.stack(np.meshgrid(np.linspace(0, 1, D),
                              np.linspace(0, 1, H),
                              np.linspace(0, 1, W), indexing='ij'), -1).reshape(-1, 3)
intensities = img.flatten()

# Random sampling
num_samples = 100000
indices = random.sample(range(len(coords)), num_samples)
sampled_coords = coords[indices]
sampled_intensities = intensities[indices]

# Dataloader
dataset = NeRPDataset(sampled_coords, sampled_intensities)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)

# Train MLP
model = NeRPMLP()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    total_loss = 0
    for x_batch, y_batch in dataloader:
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} - Loss: {total_loss / len(dataloader):.6f}")
